In [2]:
#!/usr/bin/env python3
"""
Generate 6 CW-injected IPTA MDC2 G2D2 datasets with ψ ∈ [0, π/5, 2π/5, 3π/5, 4π/5, π].
Each set: new tim/par files + one .pkl with all Pulsar objects.
"""

import numpy as np
import libstempo as lt
import os, glob, pickle
from pathlib import Path
from enterprise.pulsar import Pulsar

# ---- constants ----
SOLAR2S = 4.925490947e-6
MPC2S   = 1.02927125e14
KPC2S   = 3.085677581e19 / 2.99792458e8
EPS = np.deg2rad(23.439291111)

def ecl_to_equ(elong, elat):
    lam, beta = elong, elat
    sin_delta = np.sin(beta)*np.cos(EPS) + np.cos(beta)*np.sin(EPS)*np.sin(lam)
    delta = np.arcsin(sin_delta)
    y = np.sin(lam)*np.cos(EPS) - np.tan(beta)*np.sin(EPS)
    x = np.cos(lam)
    alpha = np.arctan2(y, x) % (2*np.pi)
    return alpha, delta

def add_cgw(psr, gwtheta, gwphi, mc, dist, fgw, phase0, psi, inc,
            pdist=1.0, pphase=None, psrTerm=True, evolve=True,
            phase_approx=False, tref=0.0):
    mc   *= SOLAR2S
    dist *= MPC2S
    w0 = np.pi*fgw
    phase0 /= 2.0
    w053 = w0**(-5/3)
    cosgwtheta, cosgwphi = np.cos(gwtheta), np.cos(gwphi)
    singwtheta, singwphi = np.sin(gwtheta), np.sin(gwphi)
    sin2psi, cos2psi = np.sin(2*psi), np.cos(2*psi)
    incfac1, incfac2 = 0.5*(3 + np.cos(2*inc)), 2*np.cos(inc)

    m     = np.array([singwphi, -cosgwphi, 0.0])
    n     = np.array([-cosgwtheta*cosgwphi, -cosgwtheta*singwphi, singwtheta])
    omhat = np.array([-singwtheta*cosgwphi, -singwtheta*singwphi, -cosgwtheta])

    if ("RAJ" in psr.pars()) and ("DECJ" in psr.pars()):
        ra, dec = psr["RAJ"].val, psr["DECJ"].val
    elif ("ELONG" in psr.pars()) and ("ELAT" in psr.pars()):
        ra, dec = ecl_to_equ(psr["ELONG"].val, psr["ELAT"].val)
    else:
        raise ValueError(f"{psr.name} lacks RAJ/DECJ or ELONG/ELAT")

    ptheta = np.pi/2 - dec
    pphi   = ra
    phat = np.array([
        np.sin(ptheta)*np.cos(pphi),
        np.sin(ptheta)*np.sin(pphi),
        np.cos(ptheta)
    ])
    fplus  = 0.5*((m@phat)**2 - (n@phat)**2)/(1 + np.dot(omhat, phat))
    fcross = ((m@phat)*(n@phat))/(1 + np.dot(omhat, phat))
    cosMu  = -np.dot(omhat, phat)

    toas = psr.toas()*86400.0 - tref
    pd = (pphase / (2*np.pi*fgw*(1 - cosMu))/KPC2S if pphase is not None else pdist) * KPC2S
    tp = toas - pd*(1 - cosMu)

    if evolve:
        fac1 = 256/5 * mc**(5/3) * w0**(8/3)
        fac2 = 1/32 / mc**(5/3)
        omega, omega_p = w0*(1 - fac1*toas)**(-3/8), w0*(1 - fac1*tp)**(-3/8)
        phase, phase_p = phase0 + fac2*(w053 - omega**(-5/3)), phase0 + fac2*(w053 - omega_p**(-5/3))
    else:
        omega = omega_p = w0
        phase, phase_p = phase0 + omega*toas, phase0 + omega*tp

    At, Bt = np.sin(2*phase)*incfac1, np.cos(2*phase)*incfac2
    At_p, Bt_p = np.sin(2*phase_p)*incfac1, np.cos(2*phase_p)*incfac2

    alpha, alpha_p = mc**(5/3)/dist/omega**(1/3), mc**(5/3)/dist/omega_p**(1/3)
    rplus   = alpha  *( At  *cos2psi + Bt  *sin2psi)
    rcross  = alpha  *(-At  *sin2psi + Bt  *cos2psi)
    rplus_p = alpha_p*( At_p*cos2psi + Bt_p*sin2psi)
    rcross_p= alpha_p*(-At_p*sin2psi + Bt_p*cos2psi)
    res = fplus*(rplus_p - rplus) + fcross*(rcross_p - rcross) if psrTerm else -fplus*rplus - fcross*rcross
    psr.stoas[:] += res/86400.0


# ---- directories ----
par_dir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par"
tim_dir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim"
base_out = Path("/scratch/na00078/projects/IPTA_MDC2/sims")
pkl_dir = Path("/scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects")
pkl_dir.mkdir(parents=True, exist_ok=True)

# ---- fixed CW params ----
gwtheta = 0.6387905062299246
gwphi   = 3.3335788713091694
mc      = 4.3e9
dist    = 75.4
fgw     = 3.7e-9
phase0  = 0.24434609527920614
inc     = 0.8412486994612669
tref    = 55443.93364609394 * 86400

# ---- ψ grid ----
psi_values = np.linspace(0, np.pi, 6)

# ---- main loop ----
for psi in psi_values:
    tag = f"psi_{psi/np.pi:.2f}pi".replace('.', 'p')
    out_dir = base_out / f"G2D2_reinj_{tag}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n=== Generating dataset for ψ = {psi:.4f} rad ({tag}) ===")

    pars = sorted(glob.glob(os.path.join(par_dir, "*.par")))
    for p in pars:
        name = os.path.basename(p)[:-4]
        t = os.path.join(tim_dir, f"{name}.tim")
        if not os.path.exists(t):
            continue
        psr = lt.tempopulsar(p, t)
        add_cgw(psr, gwtheta, gwphi, mc, dist, fgw, phase0, psi, inc,
                psrTerm=True, evolve=True, tref=tref)

        # convert to string for libstempo
        psr.savetim(str(out_dir / f"{name}.tim"))
        os.system(f"cp {p} {str(out_dir / f'{name}.par')}")

    # Build .pkl for this ψ
    pars_new = sorted(Path(out_dir).glob("*.par"))
    pulsars, skipped = [], []
    for par in pars_new:
        tim = out_dir / (par.stem + ".tim")
        try:
            psr = Pulsar(str(par), str(tim), timing_package="tempo2")
            pulsars.append(psr)
        except Exception as e:
            skipped.append((par.name, str(e)))

    pkl_name = f"G2D2_simulated_all_pulsars_{tag}.pkl"
    pkl_out = pkl_dir / pkl_name
    with open(pkl_out, "wb") as f:
        pickle.dump(pulsars, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved {len(pulsars)} pulsars -> {pkl_out}")

    if skipped:
        print("Skipped:")
        for name, reason in skipped:
            print(f"  ✖ {name} -> {reason}")

print("\nAll ψ variants completed.")



=== Generating dataset for ψ = 0.0000 rad (psi_0p00pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_0p00pi.pkl

=== Generating dataset for ψ = 0.6283 rad (psi_0p20pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_0p20pi.pkl

=== Generating dataset for ψ = 1.2566 rad (psi_0p40pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_0p40pi.pkl

=== Generating dataset for ψ = 1.8850 rad (psi_0p60pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_0p60pi.pkl

=== Generating dataset for ψ = 2.5133 rad (psi_0p80pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_0p80pi.pkl

=== Generating dataset for ψ = 3.1416 rad (psi_1p00pi) ===


Saved 33 pulsars -> /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_simulated_all_pulsars_psi_1p00pi.pkl

All ψ variants completed.
